# Selección SOAP + FPS de configuraciones entre la rotura B–C y el colapso del MLIP

Parte de los `frames.extxyz` de las dinámicas hechas con el modelo afinado (FT) que colapsaron. En cada trayectoria:

1. calcula, frame a frame, la fracción de enlaces B–C intactos, la T instantánea (de los momentos), la energía potencial, d_min, la ondulación (z_rms) y el área;
2. toma el rango **[primera rotura B–C persistente, último frame antes del colapso]**;
3. describe cada configuración con SOAP (media y dispersión por elemento), estandariza por bloques, reduce con PCA y elige las más diversas con FPS;
4. dibuja el rango de cada trayectoria, la curva de FPS y el mapa UMAP;
5. exporta las elegidas (listas para el SCF de QE), las no elegidas del rango (posible test) y un CSV con todo.

**Uso:** deja el cuaderno en la carpeta de los `frames*.extxyz`, revisa los parámetros de la sección 1 y ejecuta todas las celdas (*Run All*). La salida va a `seleccion_postrotura/`.

**Dependencias:** ase, dscribe (≥ 2.0), scikit-learn, umap-learn, matplotlib, pandas (`%pip install dscribe umap-learn` si faltan).

## 0. Importaciones

In [ ]:
# %pip install dscribe umap-learn      # descomenta si faltan
import glob
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from ase.io import read, write
from ase.neighborlist import neighbor_list
from ase.geometry import find_mic

## 1. Parámetros

Lo normal es tocar solo `TRAYECTORIAS`, `XYZ_TRAIN`, `N_SELECCION` y, si quieres repartir por trayectoria, `CUOTAS`. Los umbrales del colapso están comprobados con tus 5 dinámicas.

In [ ]:
# --- Datos -------------------------------------------------------------------
# Trayectorias de entrada (frames.extxyz de cada dinámica). Admite una lista de rutas.
TRAYECTORIAS = sorted(glob.glob("frames*.extxyz"))
# Dataset con el que ya entrenaste (opcional; uno o varios .xyz/.extxyz).
# Si lo das, el FPS parte de él: elige lo que está más lejos de lo que ya tienes,
# y aparece de fondo en el UMAP. None para no usarlo.
XYZ_TRAIN = None            # p. ej. "train_v32_filtrado.xyz" o ["train.xyz", "test.xyz"]

# --- Selección -----------------------------------------------------------------
N_SELECCION = 200           # nº total de configuraciones a elegir
CUOTAS = None               # None = FPS global; o {"frames_0": 40, "frames_1": 30, ...}
                            #   (clave = nombre del fichero sin extensión)

# --- Inicio del rango: rotura B–C ------------------------------------------------
D_ENLACE_REF = 1.9          # Å: pares B–C enlazados en el primer frame (la red intacta)
D_ROTO = 2.2                # Å: un B–C de referencia está roto si d > D_ROTO (como en la rampa)
PERSISTENCIA = 3            # frames seguidos con algún B–C roto para dar la rotura por empezada
PS_ANTES_ROTURA = 0.0       # ps antes de la primera rotura que también entran (0 = solo tras la rotura)

# --- Fin del rango: colapso ------------------------------------------------------
# Colapso = primer frame que cumple CUALQUIERA de estos criterios. Entran los frames
# anteriores a él. Antes de la rotura, en tus 5 dinámicas: |T−T_obj| ≤ 3.7σ, z_rms ≤ 3.1 Å,
# ΔE_pot ≤ 0.07 eV/át por frame; en el colapso: >40σ, z_rms > 5 Å, ΔE_pot > 1 eV/át.
Z_T_MAX = 50.0               # |T_inst − T_obj| > Z_T_MAX·σ_canónica, σ = T_obj·√(2/(3N−3))
Z_RMS_MAX = 8.0             # Å: ondulación desmedida
SALTO_EPOT = 1           # eV/át: subida de E_pot entre dos frames seguidos
D_MIN_COLAPSO = 0.8         # Å: dos átomos demasiado cerca
DESCARTAR_ANTES_COLAPSO = 0 # frames adicionales que se quitan justo antes del colapso (margen)

# --- Descriptor SOAP -----------------------------------------------------------------
ESPECIES = ["B", "C"]
SOAP_RCUT = 5.0             # Å
SOAP_NMAX = 6
SOAP_LMAX = 4
SOAP_SIGMA = 0.5            # Å
DESCRIPTOR = "media+dispersion"   # "media": SOAP medio por elemento (como en los scripts
                                  #   anteriores); "media+dispersion": añade la desviación típica
                                  #   por elemento, que recoge los pocos átomos con entornos raros
                                  #   (un B–B, un B con 4 vecinos) que el promedio diluye
VARIANZA_PCA = 0.99         # fracción de varianza que conserva el PCA

# --- UMAP -------------------------------------------------------------------------------
UMAP_VECINOS = 15
UMAP_MIN_DIST = 0.1
REFERENCIA_CADA = 10        # en el UMAP, 1 de cada n frames anteriores al rango como referencia
                            #   (gris; no son candidatas). None para no dibujarlos
SEMILLA = 42

# --- Salida -------------------------------------------------------------------------------
CARPETA_SALIDA = "seleccion_postrotura_new"
CONFIG_TYPE = "postrotura_FT"
C_SALIDA = 20.0             # Å: altura de celda de las exportadas (como tu dataset); None = la del frame
VACIO_MIN = 10.0            # Å: vacío mínimo; si espesor + VACIO_MIN > C_SALIDA, se agranda c
C_UNICA = True              # True: la misma c para todas (la mayor necesaria, redondeada a 0.5 Å);
                            #   False: cada configuración con la suya
GUARDAR_FIGURAS = True

## 2. Funciones: lectura y métricas por frame

In [ ]:
KB = 8.617333262e-5         # eV/K


def etiqueta(path, frames):
    """Nombre del fichero + T inicial y velocidad de la rampa (de T_obj_K y t_ps)."""
    nombre = Path(path).stem
    try:
        t0, t1 = frames[0].info["t_ps"], frames[1].info["t_ps"]
        T0, T1 = frames[0].info["T_obj_K"], frames[1].info["T_obj_K"]
        v = (T1 - T0) / (t1 - t0)
        return nombre, f"{nombre} ({T0:.0f} K, {v:g} K/ps)"
    except (KeyError, IndexError, ZeroDivisionError):
        return nombre, nombre


def coordenada_z(at):
    """z relativo a la mediana, sin cortes por la periodicidad en z."""
    c = at.cell[2, 2]
    z = at.positions[:, 2]
    return (z - np.median(z) + c / 2) % c - c / 2


def metricas(frames):
    """Una fila por frame con las magnitudes que definen el rango."""
    at0 = frames[0]
    sym = np.array(at0.get_chemical_symbols())
    i, j = neighbor_list("ij", at0, D_ENLACE_REF)
    m = (sym[i] == "B") & (sym[j] == "C")
    bi, bj = i[m], j[m]
    if len(bi) == 0:
        raise ValueError("No hay enlaces B–C en el primer frame: revisa D_ENLACE_REF.")
    filas, e_prev = [], np.nan
    for k, at in enumerate(frames):
        n = len(at)
        D, _ = find_mic(at.positions[bj] - at.positions[bi], at.cell, pbc=True)
        bc = float(np.mean(np.linalg.norm(D, axis=1) < D_ROTO))
        dd = neighbor_list("d", at, 1.6)
        dmin = float(dd.min()) if len(dd) else np.nan
        T_obj = float(at.info.get("T_obj_K", np.nan))
        if at.has("momenta"):
            T = 2 * at.get_kinetic_energy() / ((3 * n - 3) * KB)   # centro de masas fijo
        else:
            T = np.nan
        sigma = T_obj * np.sqrt(2.0 / (3 * n - 3))
        epot = float(at.info.get("Epot_eV_at", np.nan))
        filas.append(dict(
            frame=k, t_ps=float(at.info.get("t_ps", k)), T_obj_K=T_obj, T_inst_K=T,
            zT=(T - T_obj) / sigma, Epot_eV_at=epot, dEpot=epot - e_prev,
            BC_intactos=bc, d_min_A=dmin, z_rms_A=float(coordenada_z(at).std()),
            area_A2_at=float(abs(np.linalg.det(at.cell[:2, :2])) / n),
            barostato=int(at.info.get("barostato", 1)),
        ))
        e_prev = epot
    return pd.DataFrame(filas)


def rango(df):
    """Índices [i_ini, i_fin] del rango y un resumen de cómo se ha decidido."""
    roto = df["BC_intactos"].values < 1 - 1e-9
    i_rot = next((k for k in np.where(roto)[0] if roto[k:k + PERSISTENCIA].all()), None)
    criterios = {
        f"|T−T_obj| > {Z_T_MAX}σ": df["zT"].abs().values > Z_T_MAX,
        f"z_rms > {Z_RMS_MAX} Å": df["z_rms_A"].values > Z_RMS_MAX,
        f"ΔE_pot > {SALTO_EPOT} eV/át": df["dEpot"].values > SALTO_EPOT,
        f"d_min < {D_MIN_COLAPSO} Å": df["d_min_A"].values < D_MIN_COLAPSO,
        "barostato apagado": df["barostato"].values == 0,
    }
    marca = np.zeros(len(df), bool)
    for v in criterios.values():
        marca |= v
    i_col = int(np.argmax(marca)) if marca.any() else None
    info = dict(i_rotura=i_rot, i_colapso=i_col,
                criterios_colapso=[c for c, v in criterios.items() if i_col is not None and v[i_col]])
    if i_rot is None:
        info["motivo"] = "sin rotura B–C"
        return None, None, info
    t = df["t_ps"].values
    i_ini = int(np.searchsorted(t, t[i_rot] - PS_ANTES_ROTURA - 1e-9))
    i_fin = (len(df) - 1) if i_col is None else i_col - 1 - DESCARTAR_ANTES_COLAPSO
    if i_col is not None and i_col <= i_ini:
        info["motivo"] = "el colapso llega antes que la rotura"
        return None, None, info
    if i_fin < i_ini:
        info["motivo"] = "rango vacío con este margen"
        return None, None, info
    info["motivo"] = "ok" if i_col is not None else "sin colapso: hasta el último frame"
    return i_ini, i_fin, info

## 3. Funciones: SOAP, PCA y FPS

In [ ]:
def crear_soap():
    from dscribe.descriptors import SOAP
    return SOAP(species=ESPECIES, r_cut=SOAP_RCUT, n_max=SOAP_NMAX, l_max=SOAP_LMAX,
                sigma=SOAP_SIGMA, periodic=True, average="off", sparse=False)


def descriptor(soap, at):
    """SOAP por átomo → [media_C, (desv_C), media_B, (desv_B), ...] por elemento."""
    a = at.copy()
    a.pbc = True            # celda con vacío ≥ 2·r_cut: equivale a la lámina aislada en z
    P = soap.create(a)
    sym = np.array(a.get_chemical_symbols())
    partes = []
    for el in ESPECIES:
        Pe = P[sym == el]
        if len(Pe) == 0:
            Pe = np.zeros((1, P.shape[1]))
        partes.append(Pe.mean(0))
        if DESCRIPTOR == "media+dispersion":
            partes.append(Pe.std(0))
    return np.concatenate(partes)


def nombres_bloques():
    b = []
    for el in ESPECIES:
        b.append(f"media_{el}")
        if DESCRIPTOR == "media+dispersion":
            b.append(f"desv_{el}")
    return b


def calcular_descriptores(lista_atoms, soap, texto=""):
    t0 = time.time()
    X = np.array([descriptor(soap, a) for a in lista_atoms])
    print(f"  SOAP {texto}: {len(X)} configuraciones, {X.shape[1]} componentes, {time.time() - t0:.0f} s")
    return X


def espacio_reducido(X_ajuste, *otros):
    """Estandariza (por componente y por bloque) y aplica PCA, ajustando con X_ajuste."""
    from sklearn.decomposition import PCA
    nb = len(nombres_bloques())
    dim = X_ajuste.shape[1] // nb
    mu, sd = X_ajuste.mean(0), X_ajuste.std(0)
    vivas = sd > 1e-10 * sd.max()                   # fuera las componentes siempre nulas
    peso = np.zeros(X_ajuste.shape[1])
    for b in range(nb):
        sl = slice(b * dim, (b + 1) * dim)
        nv = max(int(vivas[sl].sum()), 1)
        peso[sl] = 1.0 / np.sqrt(nv)                # cada bloque aporta la misma varianza
    def transformar(X):
        return ((X - mu)[:, vivas] / sd[vivas]) * peso[vivas]
    pca = PCA(n_components=VARIANZA_PCA, svd_solver="full", random_state=SEMILLA)
    Z_ajuste = pca.fit_transform(transformar(X_ajuste))
    print(f"  PCA: {pca.n_components_} componentes ({100 * VARIANZA_PCA:.0f} % de la varianza), "
          f"{int(vivas.sum())} de {len(vivas)} componentes SOAP no nulas")
    return [Z_ajuste] + [pca.transform(transformar(X)) if X is not None and len(X) else None
                         for X in otros]


def fps(Z, n, Z_ref=None):
    """FPS clásico (máx-mín). Sin referencia empieza por la más cercana al centroide;
    con referencia (el train) la distancia inicial es la distancia al train.
    Devuelve el orden elegido y la distancia máx-mín en el momento de elegir cada una."""
    n = min(n, len(Z))
    if Z_ref is not None and len(Z_ref):
        dmin = np.full(len(Z), np.inf)
        for a in range(0, len(Z_ref), 2000):        # por trozos, sin matrices enormes
            d = np.linalg.norm(Z[:, None, :] - Z_ref[None, a:a + 2000, :], axis=2)
            dmin = np.minimum(dmin, d.min(1))
        primero = int(np.argmax(dmin))
        d0 = float(dmin[primero])
    else:
        primero = int(np.argmin(np.linalg.norm(Z - Z.mean(0), axis=1)))
        dmin = np.full(len(Z), np.inf)
        d0 = np.nan
    orden, dist = [primero], [d0]
    dmin = np.minimum(dmin, np.linalg.norm(Z - Z[primero], axis=1))
    for _ in range(n - 1):
        k = int(np.argmax(dmin))
        orden.append(k)
        dist.append(float(dmin[k]))
        dmin = np.minimum(dmin, np.linalg.norm(Z - Z[k], axis=1))
    return np.array(orden), np.array(dist)

## 4. Funciones: exportar

In [ ]:
def espesor(at):
    dz = coordenada_z(at)
    return float(dz.max() - dz.min())


def para_qe(at, df_fila, nombre, rango_fps, d_fps, c_fija=None):
    """Copia limpia para el SCF: sin momentos, z centrado y celda c = C_SALIDA (o mayor).
    Con c_fija, usa esa c (la común a todas)."""
    a = at.copy()
    a.calc = None
    for clave in list(a.arrays):
        if clave not in ("numbers", "positions"):
            del a.arrays[clave]
    cell = a.cell.array.copy()
    if not (abs(cell[0, 2]) < 1e-8 and abs(cell[1, 2]) < 1e-8 and
            abs(cell[2, 0]) < 1e-8 and abs(cell[2, 1]) < 1e-8):
        raise ValueError("La celda no tiene el eje c perpendicular al plano.")
    dz = coordenada_z(a)
    esp = float(dz.max() - dz.min())
    if c_fija is not None:
        c_nueva = c_fija
    else:
        c_nueva = cell[2, 2] if C_SALIDA is None else max(C_SALIDA, esp + VACIO_MIN)
    a.positions[:, 2] = dz - (dz.max() + dz.min()) / 2 + c_nueva / 2
    cell[2] = [0.0, 0.0, c_nueva]
    a.set_cell(cell, scale_atoms=False)
    a.pbc = True
    a.wrap()
    a.info = dict(
        config_type=CONFIG_TYPE, traj=nombre, frame=int(df_fila["frame"]),
        t_ps=float(df_fila["t_ps"]), T_obj_K=float(df_fila["T_obj_K"]),
        T_inst_K=float(df_fila["T_inst_K"]), BC_intactos=float(df_fila["BC_intactos"]),
        area_A2_at=float(df_fila["area_A2_at"]), z_rms_A=float(df_fila["z_rms_A"]),
        d_min_A=float(df_fila["d_min_A"]), espesor_A=esp,
    )
    if np.isfinite(df_fila.get("ps_antes_colapso", np.nan)):
        a.info["ps_antes_colapso"] = float(df_fila["ps_antes_colapso"])
    if rango_fps is not None:
        a.info["fps_rank"] = int(rango_fps)
        if np.isfinite(d_fps):          # la primera elegida sin train no tiene distancia
            a.info["d_fps"] = float(d_fps)
    return a

## 5. Estilo de las figuras

In [ ]:
TINTA, TINTA_2, TINTA_MUDA = "#0b0b0b", "#52514e", "#898781"
REJILLA, EJE, SUPERFICIE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
AZUL, NARANJA = "#2a78d6", "#eb6834"
GRIS_FONDO, GRIS_REF = "#d9d8d2", "#a9a79f"
CMAP_AZUL = LinearSegmentedColormap.from_list(
    "azul", ["#b7d3f6", "#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"])
CMAP_NARANJA = LinearSegmentedColormap.from_list(
    "naranja", ["#fbd9c9", "#f6ad8b", "#f08a5f", "#eb6834", "#c4501f", "#9a3c14", "#6f2a0c"])

plt.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "savefig.facecolor": SUPERFICIE,
    "axes.edgecolor": EJE, "axes.labelcolor": TINTA_2, "axes.titlecolor": TINTA,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": TINTA_MUDA, "ytick.color": TINTA_MUDA,
    "xtick.labelcolor": TINTA_2, "ytick.labelcolor": TINTA_2,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
    "legend.frameon": False, "lines.linewidth": 1.5,
})


def guardar(fig, nombre):
    if GUARDAR_FIGURAS:
        fig.savefig(Path(CARPETA_SALIDA) / f"{nombre}.png", dpi=200, bbox_inches="tight")
        fig.savefig(Path(CARPETA_SALIDA) / f"{nombre}.pdf", bbox_inches="tight")

## 6. Lectura, métricas y rango de cada trayectoria

Para cada trayectoria imprime dónde empieza la rotura, dónde colapsa (y por qué criterio) y el rango que queda.

In [ ]:
Path(CARPETA_SALIDA).mkdir(parents=True, exist_ok=True)
if not TRAYECTORIAS:
    raise FileNotFoundError("No hay trayectorias: revisa TRAYECTORIAS.")

trayectorias = {}       # nombre → dict(frames, df, i_ini, i_fin, info, etiqueta)
print("Lectura y rango por trayectoria")
for path in TRAYECTORIAS:
    t0 = time.time()
    frames = read(path, index=":")
    nombre, etq = etiqueta(path, frames)
    if nombre in trayectorias:
        raise ValueError(f"Nombre repetido: {nombre}. Renombra los ficheros.")
    df = metricas(frames)
    i_ini, i_fin, info = rango(df)
    trayectorias[nombre] = dict(frames=frames, df=df, i_ini=i_ini, i_fin=i_fin,
                                info=info, etiqueta=etq, path=str(path))
    t, To = df["t_ps"].values, df["T_obj_K"].values
    if i_ini is None:
        print(f"  {etq}: {len(df)} frames — se salta ({info['motivo']})")
        continue
    txt_col = (f"colapso en t = {t[info['i_colapso']]:.1f} ps (T_obj {To[info['i_colapso']]:.0f} K; "
               f"{', '.join(info['criterios_colapso'])})" if info["i_colapso"] is not None
               else "sin colapso")
    print(f"  {etq}: {len(df)} frames, {time.time() - t0:.0f} s\n"
          f"      rotura en t = {t[info['i_rotura']]:.1f} ps (T_obj {To[info['i_rotura']]:.0f} K); {txt_col}\n"
          f"      rango: t = {t[i_ini]:.1f}–{t[i_fin]:.1f} ps (T_obj {To[i_ini]:.0f}–{To[i_fin]:.0f} K), "
          f"{i_fin - i_ini + 1} frames; último: |T−T_obj| = {abs(df['zT'][i_fin]):.1f}σ, "
          f"z_rms {df['z_rms_A'][i_fin]:.2f} Å, B–C intactos {100 * df['BC_intactos'][i_fin]:.0f} %")

# Candidatas (dentro del rango) y referencia (antes del rango, solo para el UMAP)
cand_atoms, cand_filas, ref_atoms, ref_filas = [], [], [], []
for nombre, tr in trayectorias.items():
    if tr["i_ini"] is None:
        continue
    i_col = tr["info"]["i_colapso"]
    t_col = tr["df"]["t_ps"][i_col] if i_col is not None else np.nan
    for k in range(tr["i_ini"], tr["i_fin"] + 1):
        cand_atoms.append(tr["frames"][k])
        fila = dict(traj=nombre, **tr["df"].iloc[k].to_dict())
        fila["ps_antes_colapso"] = t_col - fila["t_ps"]     # NaN si no colapsó
        cand_filas.append(fila)
    if REFERENCIA_CADA:
        for k in range(0, tr["i_ini"], REFERENCIA_CADA):
            ref_atoms.append(tr["frames"][k])
            ref_filas.append(dict(traj=nombre, **tr["df"].iloc[k].to_dict()))
cand = pd.DataFrame(cand_filas)
ref = pd.DataFrame(ref_filas)
if cand.empty:
    raise RuntimeError("Ninguna trayectoria tiene configuraciones en el rango.")
print(f"\nCandidatas: {len(cand)} configuraciones de {cand['traj'].nunique()} trayectorias "
      f"(referencia para el UMAP: {len(ref)})")

train_atoms = []
if XYZ_TRAIN:
    for p in ([XYZ_TRAIN] if isinstance(XYZ_TRAIN, (str, Path)) else XYZ_TRAIN):
        train_atoms += read(p, index=":")
    print(f"Train: {len(train_atoms)} configuraciones")

## 7. SOAP, PCA y FPS

La resolución del FPS es la distancia máx-mín de la última elegida; se compara con la distancia entre frames seguidos (1 ps).

In [ ]:
print("\nDescriptores")
soap = crear_soap()
X_cand = calcular_descriptores(cand_atoms, soap, "candidatas")
X_train = calcular_descriptores(train_atoms, soap, "train") if train_atoms else None
X_ref = calcular_descriptores(ref_atoms, soap, "referencia") if ref_atoms else None

X_ajuste = X_cand if X_train is None else np.vstack([X_cand, X_train])
Z_ajuste, Z_ref = espacio_reducido(X_ajuste, X_ref)
Z_cand = Z_ajuste[:len(X_cand)]
Z_train = Z_ajuste[len(X_cand):] if X_train is not None else None

# Resolución de referencia: distancia entre frames seguidos (1 ps) de una misma trayectoria
d_seguidos = []
for nombre in cand["traj"].unique():
    idx = np.where(cand["traj"].values == nombre)[0]
    if len(idx) > 1:
        d_seguidos += list(np.linalg.norm(np.diff(Z_cand[idx], axis=0), axis=1))
d_seguidos = float(np.median(d_seguidos)) if d_seguidos else np.nan

# FPS: orden completo (para la curva) y selección
orden_global, dist_global = fps(Z_cand, len(Z_cand), Z_train)
cand["fps_rank_global"] = -1
cand.loc[orden_global, "fps_rank_global"] = np.arange(len(orden_global))
cand["d_fps_global"] = np.nan
cand.loc[orden_global, "d_fps_global"] = dist_global

cand["seleccionada"] = False
cand["fps_rank"] = -1
cand["d_fps"] = np.nan
if CUOTAS is None:
    elegidas = orden_global[:N_SELECCION]
    cand.loc[elegidas, "seleccionada"] = True
    cand.loc[elegidas, "fps_rank"] = np.arange(len(elegidas))
    cand.loc[elegidas, "d_fps"] = dist_global[:len(elegidas)]
else:
    for nombre, cupo in CUOTAS.items():
        idx = np.where(cand["traj"].values == nombre)[0]
        if len(idx) == 0:
            print(f"  aviso: {nombre} no tiene candidatas")
            continue
        o, d = fps(Z_cand[idx], cupo, Z_train)
        cand.loc[idx[o], "seleccionada"] = True
        cand.loc[idx[o], "fps_rank"] = np.arange(len(o))
        cand.loc[idx[o], "d_fps"] = d
n_sel = int(cand["seleccionada"].sum())
res_fps = float(cand.loc[cand["seleccionada"], "d_fps"].min())
print(f"\nFPS: {n_sel} elegidas de {len(cand)}; resolución (d de la última elegida) = {res_fps:.2f}; "
      f"frames seguidos (1 ps) = {d_seguidos:.2f}")
print(cand.groupby("traj").agg(rango=("frame", "size"), elegidas=("seleccionada", "sum")).to_string())
ultimos = cand["ps_antes_colapso"] <= 20
if ultimos.any():
    print(f"  En los últimos 20 ps antes del colapso: {int(ultimos.sum())} candidatas "
          f"({100 * ultimos.mean():.0f} %), {int((ultimos & cand['seleccionada']).sum())} elegidas "
          f"({100 * (ultimos & cand['seleccionada']).sum() / max(n_sel, 1):.0f} %)")

## 8. UMAP

La primera vez tarda unos segundos más (umap compila con numba).

In [ ]:
print("\nUMAP")
import warnings  # noqa: E402
import umap  # noqa: E402  (tarda unos segundos la primera vez: compila con numba)
bloques_umap = [Z_cand] + ([Z_train] if Z_train is not None else []) + ([Z_ref] if Z_ref is not None else [])
opciones_umap = dict(n_neighbors=UMAP_VECINOS, min_dist=UMAP_MIN_DIST, random_state=SEMILLA)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)       # aviso de n_jobs al fijar la semilla
    try:                                               # umap-learn ≥ 0.6: el algoritmo de 0.5.x
        reductor = umap.UMAP(compatibility_layout=True, **opciones_umap)
    except TypeError:                                  # umap-learn 0.5.x
        reductor = umap.UMAP(**opciones_umap)
    emb = reductor.fit_transform(np.vstack(bloques_umap))
E_cand = emb[:len(Z_cand)]
cand["umap_1"], cand["umap_2"] = E_cand[:, 0], E_cand[:, 1]
pos = len(Z_cand)
E_train = None
if Z_train is not None:
    E_train = emb[pos:pos + len(Z_train)]
    pos += len(Z_train)
E_ref = emb[pos:] if Z_ref is not None else None

## 9. Figura: rango de cada trayectoria

In [ ]:
con_rango = [n for n, tr in trayectorias.items() if tr["i_ini"] is not None]
fig, axes = plt.subplots(len(con_rango), 2, figsize=(10, 2.2 * len(con_rango)), squeeze=False)
for fila, nombre in enumerate(con_rango):
    tr = trayectorias[nombre]
    df, i_ini, i_fin, i_col = tr["df"], tr["i_ini"], tr["i_fin"], tr["info"]["i_colapso"]
    a_ver = max(0, i_ini - int(1.5 * (i_fin - i_ini + 1)) - 20)
    b_ver = min(len(df) - 1, (i_col if i_col is not None else i_fin) + 3)
    tramo = df.iloc[a_ver:b_ver + 1]
    sel = cand[(cand["traj"] == nombre) & cand["seleccionada"]]
    for col, (var, esc, ylab) in enumerate([("BC_intactos", 100, "B–C intactos (%)"),
                                            ("area_A2_at", 1, "Área (Å²/át)")]):
        ax = axes[fila, col]
        ax.axvspan(df["T_obj_K"][i_ini], df["T_obj_K"][i_fin], color=REJILLA, alpha=0.6, lw=0)
        ax.plot(tramo["T_obj_K"], esc * tramo[var], color=AZUL, lw=1.2)
        ax.scatter(sel["T_obj_K"], esc * sel[var], s=22, color=NARANJA, edgecolor=SUPERFICIE,
                   linewidth=1.0, zorder=3)
        if i_col is not None:
            ax.axvline(df["T_obj_K"][i_col], color=TINTA_2, lw=1.0)
        ax.set_ylabel(ylab)
        if col == 0:
            ax.set_title(tr["etiqueta"], loc="left")
        if fila == len(con_rango) - 1:
            ax.set_xlabel("T objetivo (K)")
from matplotlib.patches import Patch  # noqa: E402
fig.legend(handles=[Line2D([], [], color=AZUL, lw=1.2, label="trayectoria"),
                    Patch(color=REJILLA, alpha=0.6, label="rango elegible"),
                    Line2D([], [], marker="o", ls="", color=NARANJA, ms=5, label="elegidas (FPS)"),
                    Line2D([], [], color=TINTA_2, lw=1.0, label="primer frame colapsado")],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.005))
fig.tight_layout(rect=(0, 0.03, 1, 1))
guardar(fig, "rango_por_trayectoria")
plt.show()

## 10. Figura: curva de FPS

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
x = np.arange(1, len(dist_global) + 1)
ax.plot(x, dist_global, color=AZUL, lw=1.5)
if np.isfinite(d_seguidos):
    ax.axhline(d_seguidos, color=TINTA_MUDA, lw=1.0)
    ax.annotate("frames seguidos (1 ps)", (x[-1], d_seguidos), xytext=(0, 3), textcoords="offset points",
                ha="right", va="bottom", color=TINTA_2, fontsize=8)
if CUOTAS is None:
    ax.axvline(N_SELECCION, color=TINTA_2, lw=1.0)
    ax.annotate(f"N = {N_SELECCION}", (N_SELECCION, 1), xycoords=("data", "axes fraction"),
                xytext=(3, -2), textcoords="offset points", ha="left", va="top", color=TINTA_2, fontsize=8)
ax.set_xlabel("Configuraciones elegidas")
ax.set_ylabel("Distancia máx-mín (PCA)")
ax.set_title("Curva de FPS" + (" (partiendo del train)" if Z_train is not None else ""), loc="left")
fig.tight_layout()
guardar(fig, "curva_fps")
plt.show()

## 11. Figura: UMAP general (color = T objetivo y B–C rotos)

In [ ]:
def fondo(ax):
    if E_train is not None:
        ax.scatter(E_train[:, 0], E_train[:, 1], s=6, color=GRIS_FONDO, lw=0, label="train", zorder=1)
    if E_ref is not None:
        ax.scatter(E_ref[:, 0], E_ref[:, 1], s=6, color=GRIS_REF, lw=0, label="antes de la rotura", zorder=1)


sel_m = cand["seleccionada"].values
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (valores, cmap, titulo, cbar_txt) in zip(axes, [
        (cand["T_obj_K"].values, CMAP_AZUL, "Color: T objetivo", "T objetivo (K)"),
        (100 * (1 - cand["BC_intactos"].values), CMAP_NARANJA, "Color: B–C rotos", "B–C rotos (%)")]):
    fondo(ax)
    sc = ax.scatter(E_cand[:, 0], E_cand[:, 1], c=valores, cmap=cmap, s=12, lw=0, zorder=2)
    ax.scatter(E_cand[sel_m, 0], E_cand[sel_m, 1], s=34, facecolor="none", edgecolor=TINTA,
               linewidth=0.9, zorder=3, label="elegidas (FPS)")
    cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(cbar_txt, color=TINTA_2)
    cb.outline.set_visible(False)
    ax.set_title(titulo, loc="left")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)
axes[0].legend(loc="best", markerscale=1.5)
fig.suptitle(f"UMAP del SOAP: {len(cand)} candidatas, {n_sel} elegidas", x=0.01, ha="left",
             color=TINTA, fontsize=11)
fig.tight_layout()
guardar(fig, "umap_general")
plt.show()

## 12. Figura: UMAP por trayectoria (small multiples)

In [ ]:
nombres = list(cand["traj"].unique())
ncol = min(3, len(nombres))
nfil = int(np.ceil(len(nombres) / ncol))
fig, axes = plt.subplots(nfil, ncol, figsize=(3.8 * ncol, 3.5 * nfil), squeeze=False)
for k, ax in enumerate(axes.flat):
    if k >= len(nombres):
        ax.axis("off")
        continue
    nombre = nombres[k]
    m = cand["traj"].values == nombre
    fondo(ax)
    ax.scatter(E_cand[~m, 0], E_cand[~m, 1], s=6, color=GRIS_FONDO, lw=0, zorder=1)
    ax.scatter(E_cand[m, 0], E_cand[m, 1], s=12, color=AZUL, lw=0, zorder=2, label="rango")
    ms = m & sel_m
    ax.scatter(E_cand[ms, 0], E_cand[ms, 1], s=30, color=NARANJA, edgecolor=SUPERFICIE,
               linewidth=1.0, zorder=3, label="elegidas")
    ax.set_title(f"{trayectorias[nombre]['etiqueta']}\n{m.sum()} en el rango, {ms.sum()} elegidas",
                 loc="left", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)
leyenda = [Line2D([], [], marker="o", ls="", color=AZUL, ms=5, label="rango de la trayectoria"),
           Line2D([], [], marker="o", ls="", color=NARANJA, ms=6, label="elegidas"),
           Line2D([], [], marker="o", ls="", color=GRIS_FONDO, ms=4, label="otras trayectorias")]
if E_ref is not None:
    leyenda.append(Line2D([], [], marker="o", ls="", color=GRIS_REF, ms=4, label="antes de la rotura"))
fig.legend(handles=leyenda, loc="lower center", ncol=len(leyenda), bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.04, 1, 1))
guardar(fig, "umap_por_trayectoria")
plt.show()

## 13. Exportar

`seleccion_fps.extxyz` va al SCF; `rango_no_elegidas.extxyz` sirve como test.

In [ ]:
salida = Path(CARPETA_SALIDA)
cand["espesor_A"] = [espesor(a) for a in cand_atoms]
c_comun = None
if C_UNICA and C_SALIDA is not None:
    c_comun = max(C_SALIDA, float(np.ceil(2 * (cand["espesor_A"].max() + VACIO_MIN)) / 2))
elegidas_atoms, descartadas_atoms = [], []
for k in range(len(cand)):
    fila = cand.iloc[k]
    at = cand_atoms[k]
    if fila["seleccionada"]:
        elegidas_atoms.append((int(fila["fps_rank"]), fila["traj"],
                               para_qe(at, fila, fila["traj"], fila["fps_rank"], fila["d_fps"], c_comun)))
    else:
        descartadas_atoms.append(para_qe(at, fila, fila["traj"], None, None, c_comun))
elegidas_atoms.sort(key=lambda x: (x[1], x[0]) if CUOTAS else x[0])   # en orden de FPS
elegidas_atoms = [a for _, _, a in elegidas_atoms]

write(salida / "seleccion_fps.extxyz", elegidas_atoms, format="extxyz")
if descartadas_atoms:
    write(salida / "rango_no_elegidas.extxyz", descartadas_atoms, format="extxyz")
cand.to_csv(salida / "candidatas.csv", index=False, encoding="utf-8")

c_usadas = sorted({round(a.cell[2, 2], 2) for a in elegidas_atoms})
resumen = dict(
    parametros=dict(
        TRAYECTORIAS=[str(p) for p in TRAYECTORIAS], XYZ_TRAIN=XYZ_TRAIN, N_SELECCION=N_SELECCION,
        CUOTAS=CUOTAS, D_ENLACE_REF=D_ENLACE_REF, D_ROTO=D_ROTO, PERSISTENCIA=PERSISTENCIA,
        PS_ANTES_ROTURA=PS_ANTES_ROTURA, Z_T_MAX=Z_T_MAX, Z_RMS_MAX=Z_RMS_MAX, SALTO_EPOT=SALTO_EPOT,
        D_MIN_COLAPSO=D_MIN_COLAPSO, DESCARTAR_ANTES_COLAPSO=DESCARTAR_ANTES_COLAPSO,
        SOAP=dict(especies=ESPECIES, r_cut=SOAP_RCUT, n_max=SOAP_NMAX, l_max=SOAP_LMAX,
                  sigma=SOAP_SIGMA, descriptor=DESCRIPTOR),
        VARIANZA_PCA=VARIANZA_PCA, C_SALIDA=C_SALIDA, VACIO_MIN=VACIO_MIN, C_UNICA=C_UNICA,
        SEMILLA=SEMILLA),
    trayectorias={
        n: dict(etiqueta=tr["etiqueta"], fichero=tr["path"], n_frames=len(tr["df"]),
                motivo=tr["info"]["motivo"],
                t_rotura_ps=None if tr["info"]["i_rotura"] is None else float(tr["df"]["t_ps"][tr["info"]["i_rotura"]]),
                T_rotura_K=None if tr["info"]["i_rotura"] is None else float(tr["df"]["T_obj_K"][tr["info"]["i_rotura"]]),
                t_colapso_ps=None if tr["info"]["i_colapso"] is None else float(tr["df"]["t_ps"][tr["info"]["i_colapso"]]),
                T_colapso_K=None if tr["info"]["i_colapso"] is None else float(tr["df"]["T_obj_K"][tr["info"]["i_colapso"]]),
                criterios_colapso=tr["info"]["criterios_colapso"],
                n_rango=int((cand["traj"] == n).sum()),
                n_elegidas=int(((cand["traj"] == n) & cand["seleccionada"]).sum()))
        for n, tr in trayectorias.items()},
    n_candidatas=len(cand), n_elegidas=n_sel, resolucion_fps=res_fps, d_frames_seguidos=d_seguidos,
    c_celda_exportadas_A=c_usadas,
)
with open(salida / "resumen.json", "w", encoding="utf-8") as f:   # utf-8: en Windows el defecto es cp1252
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print(f"\nGuardado en {salida}/:")
print(f"  seleccion_fps.extxyz      {len(elegidas_atoms)} configuraciones para el SCF "
      f"(config_type={CONFIG_TYPE}; c = {', '.join(f'{c:g}' for c in c_usadas)} Å)")
print(f"  rango_no_elegidas.extxyz  {len(descartadas_atoms)} (posible test)")
print("  candidatas.csv, resumen.json y las figuras (.png y .pdf)")